# Oversized AOI — local tiling + mosaic

An AOI larger than the 2500 px Process cap is split into tiles, each rendered via Process, then mosaicked locally with `pyramids.dataset.merge.merge_rasters` into one GeoTIFF — **no S3 bucket needed**. Force it with `api='tiling'` (or let auto-routing pick it when no `batch_output` is set).

## Credentials guard

In [ ]:
import os
from pathlib import Path


def has_sh_credentials() -> bool:
    """Whether Sentinel Hub OAuth client-credentials are available."""
    return bool(os.environ.get("SENTINELHUB_CLIENT_ID") and os.environ.get("SENTINELHUB_CLIENT_SECRET"))


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    Sentinel Hub credentials are configured (so the docs build never needs
    secrets). Set SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET to run the cell for real.
    """
    if not has_sh_credentials():
        print(
            "No Sentinel Hub credentials found - skipping the live download.\n"
            "Mint an OAuth client_credentials pair in the CDSE Dashboard and set\n"
            "SENTINELHUB_CLIENT_ID / SENTINELHUB_CLIENT_SECRET (see the Authentication page)."
        )
        return []
    results = facade.download(**download_kwargs)
    for item in results:
        print(item)
    return results


## A wide bbox via the tiling plane

In [ ]:
from earthlens import EarthLens

facade = EarthLens(
    data_source='sentinel-hub',
    dataset='sentinel-2-l2a-ndvi', variables=[],
    start='2020-06-10', end='2020-06-20',
    aoi=[14.0, 40.6, 14.5, 41.1],
    path='data/sh-tiling', resolution=20,
    api='tiling',
)
paths = run_or_skip(facade)
paths